# OrthoAI ACL Injury Risk Prediction Pipeline
## A Machine Learning Analysis of Biomechanical and Training Load Predictors

---

**Author:** Muhammad Ebadur Rahman Siddiqui  
**Platform:** OrthoAI Research (orthoai-research.github.io)  
**Dataset:** Synthetic ACL Biomechanical Dataset v1.0 — 1,500 athlete-seasons  
**Libraries:** pandas · numpy · scikit-learn · matplotlib · seaborn  

---

### Abstract

Anterior cruciate ligament (ACL) injuries represent one of the most devastating
musculoskeletal injuries in competitive sport, carrying significant physical,
psychological, and economic costs. This analysis applies supervised machine learning
to a multi-factor biomechanical dataset to identify the strongest predictors of
ACL injury risk and evaluate the discriminative performance of three classifiers:
Logistic Regression, Random Forest, and Gradient Boosting.

**Key findings:** An Acute:Chronic Workload Ratio (ACWR) >1.5, elevated knee
valgus angle, and prior ACL history emerged as the strongest predictors.
The best model achieved a Test AUC of 0.709, consistent with published ML
models in sports injury epidemiology literature.


## 1. Setup & Imports

In [ ]:

import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot  as plt
import matplotlib.gridspec as gridspec
import seaborn             as sns

from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.impute            import SimpleImputer
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics           import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline   import Pipeline
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# Plotting style
ACCENT   = "#e94560"
LIGHT_BG = "#f7f9fc"
DARK     = "#1a1a2e"
plt.rcParams.update({
    "figure.facecolor":  LIGHT_BG,
    "axes.facecolor":    LIGHT_BG,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
})
print("✓ Imports loaded successfully")


## 2. Load & Audit the Dataset

The dataset contains **1,500 athlete-seasons** across 5 sports (Football, Basketball,
Rugby, Handball, Volleyball). Each row represents one athlete over one competitive season.
Features were derived from published biomechanical and sports medicine literature.


In [ ]:

df = pd.read_csv("data/acl_injury_dataset.csv")

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nTarget (ACL Injury) distribution:")
print(df["acl_injury_this_season"].value_counts())
print(f"\nInjury prevalence: {df['acl_injury_this_season'].mean()*100:.1f}%")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum()>0])


In [ ]:

# Preview the data
df.head(10)


In [ ]:

# Statistical summary of key clinical variables
df[["acwr","knee_valgus_deg","hq_ratio","hop_symmetry_pct","sleep_hours"]].describe().round(3)


## 3. Exploratory Data Analysis (EDA)

Before building any model, we **explore the data** to understand distributions,
spot patterns, and identify which features separate injured from non-injured athletes.
This is what separates real research from guesswork.


In [ ]:

TARGET     = "acl_injury_this_season"
injured    = df[df[TARGET] == 1]
not_injured= df[df[TARGET] == 0]

print(f"Injured athletes     : {len(injured)}")
print(f"Non-injured athletes : {len(not_injured)}")


### 3.1 Injury Overview Dashboard

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("OrthoAI ACL Dataset — Overview Dashboard", fontsize=15, fontweight="bold")

# Prevalence pie
axes[0,0].pie([len(not_injured), len(injured)],
              labels=["No Injury","ACL Injury"],
              colors=["#aed6f1", ACCENT], autopct="%1.1f%%", startangle=90)
axes[0,0].set_title("Season Injury Prevalence")

# Sport
sport_inj = df.groupby("sport")[TARGET].mean().sort_values(ascending=False)
sport_inj.plot(kind="bar", ax=axes[0,1], color=ACCENT, edgecolor="white", width=0.65)
axes[0,1].set_title("Injury Rate by Sport")
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=30, ha="right")

# Sex
sex_inj = df.groupby("sex")[TARGET].mean()
sex_inj.plot(kind="bar", ax=axes[0,2], color=["#5dade2", ACCENT], edgecolor="white")
axes[0,2].set_title("Injury Rate by Sex"); axes[0,2].set_xticklabels(sex_inj.index, rotation=0)

# ACWR
axes[1,0].hist(not_injured["acwr"], bins=35, alpha=0.65, color="#5dade2", label="No Injury")
axes[1,0].hist(injured["acwr"],     bins=35, alpha=0.75, color=ACCENT,    label="ACL Injury")
axes[1,0].axvline(1.5, color=DARK, linestyle="--", label="Danger Zone")
axes[1,0].set_title("ACWR Distribution"); axes[1,0].legend(fontsize=8)

# Knee Valgus
axes[1,1].hist(not_injured["knee_valgus_deg"], bins=35, alpha=0.65, color="#5dade2", label="No Injury")
axes[1,1].hist(injured["knee_valgus_deg"],     bins=35, alpha=0.75, color=ACCENT,    label="ACL Injury")
axes[1,1].set_title("Knee Valgus Angle"); axes[1,1].legend(fontsize=8)

# H:Q ratio
axes[1,2].hist(not_injured["hq_ratio"], bins=35, alpha=0.65, color="#5dade2", label="No Injury")
axes[1,2].hist(injured["hq_ratio"],     bins=35, alpha=0.75, color=ACCENT,    label="ACL Injury")
axes[1,2].axvline(0.60, color=DARK, linestyle="--", label="Risk Threshold")
axes[1,2].set_title("H:Q Ratio"); axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.show()


### 3.2 Correlation Heatmap

In [ ]:

NUMERIC_FEATURES = [
    "age","bmi","weekly_training_hours","acwr","session_rpe",
    "monotony_index","knee_valgus_deg","hop_symmetry_pct","hq_ratio",
    "landing_force_bw","sleep_hours","wellness_score"
]
corr_df = df[NUMERIC_FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlBu_r", center=0, square=True,
            linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Top correlations with target
print("\nTop correlations with ACL injury:")
print(corr_df[TARGET].sort_values(ascending=False).to_string())


### 3.3 Risk Factor Boxplots

In [ ]:

key_features = ["acwr","knee_valgus_deg","hq_ratio","hop_symmetry_pct","sleep_hours","landing_force_bw"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, feat in enumerate(key_features):
    d0 = df[df[TARGET]==0][feat].dropna()
    d1 = df[df[TARGET]==1][feat].dropna()
    bp = axes[i].boxplot([d0, d1], labels=["No Injury","ACL Injury"],
                          patch_artist=True, medianprops={"color":"black","linewidth":2})
    bp["boxes"][0].set_facecolor("#aed6f1")
    bp["boxes"][1].set_facecolor(ACCENT)
    axes[i].set_title(feat.replace("_"," ").title())
fig.suptitle("Key Risk Factors: Injured vs Non-Injured", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 4. Feature Engineering

We encode categorical variables and create **clinically-grounded binary flags**
(e.g., ACWR > 1.5 is the published danger zone for injury). These composite features
help the model learn non-linear risk thresholds.


In [ ]:

df_ml = df.copy()

# Encode sex and sport as numbers
le_sex   = LabelEncoder()
le_sport = LabelEncoder()
df_ml["sex_enc"]   = le_sex.fit_transform(df_ml["sex"])
df_ml["sport_enc"] = le_sport.fit_transform(df_ml["sport"])

# Clinical risk flags
df_ml["high_acwr"]      = (df_ml["acwr"]              > 1.5).astype(int)
df_ml["poor_hq_ratio"]  = (df_ml["hq_ratio"]          < 0.60).astype(int)
df_ml["limb_asymmetry"] = (df_ml["hop_symmetry_pct"]  < 90).astype(int)
df_ml["high_valgus"]    = (df_ml["knee_valgus_deg"]   > 8).astype(int)
df_ml["sleep_deprived"] = (df_ml["sleep_hours"]       < 6).astype(int)
df_ml["composite_risk"] = (
    df_ml["high_acwr"] + df_ml["poor_hq_ratio"] +
    df_ml["limb_asymmetry"] + df_ml["high_valgus"]
)

BINARY_FEATURES = ["prev_knee_injury","prev_acl_injury"]
FEATURE_COLS = (
    NUMERIC_FEATURES + BINARY_FEATURES +
    ["sex_enc","sport_enc","high_acwr","poor_hq_ratio",
     "limb_asymmetry","high_valgus","sleep_deprived","composite_risk"]
)

X = df_ml[FEATURE_COLS]
y = df_ml[TARGET]

print(f"Feature matrix: {X.shape[0]} samples × {X.shape[1]} features")

# Train/test split — stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# Composite risk flag distribution
print("\nComposite risk score distribution:")
print(df_ml["composite_risk"].value_counts().sort_index())


## 5. Model Training

Three classifiers are trained using **sklearn Pipelines** (imputation → scaling → model).
We use **5-fold stratified cross-validation** on the training set to get unbiased AUC estimates,
then evaluate final performance on the held-out test set.


In [ ]:

models = {
    "Logistic Regression": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("clf",    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf",    RandomForestClassifier(n_estimators=300, max_depth=8,
                       class_weight="balanced", min_samples_leaf=5, random_state=42, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf",    GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                       max_depth=4, subsample=0.8, random_state=42))
    ]),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"{'Model':<30} {'CV AUC':>10} {'Std':>8} {'Test AUC':>10}")
print("-" * 62)
for name, pipe in models.items():
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    pipe.fit(X_train, y_train)
    test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:,1])
    results[name] = {"cv_auc": cv_auc.mean(), "cv_std": cv_auc.std(),
                     "test_auc": test_auc, "pipe": pipe}
    print(f"{name:<30} {cv_auc.mean():>10.3f} {cv_auc.std():>8.3f} {test_auc:>10.3f}")

best_name = max(results, key=lambda k: results[k]["test_auc"])
best_pipe = results[best_name]["pipe"]
print(f"\n★ Best model: {best_name}  (Test AUC = {results[best_name]['test_auc']:.3f})")


## 6. Model Evaluation

### What is AUC-ROC?
AUC (Area Under the ROC Curve) measures how well the model distinguishes injured
from non-injured athletes. **0.5 = random chance. 1.0 = perfect.** Published ML
models for ACL/sports injury typically range from 0.65–0.80 in external validation.


In [ ]:

y_prob = best_pipe.predict_proba(X_test)[:,1]
y_pred = best_pipe.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Injury","ACL Injury"]))


In [ ]:

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ROC curves
ax_roc = fig.add_subplot(gs[0,0])
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["pipe"].predict_proba(X_test)[:,1])
    ax_roc.plot(fpr, tpr, linewidth=2, label=f"{name} ({res['test_auc']:.3f})")
ax_roc.plot([0,1],[0,1],"--", color="#aaa"); ax_roc.set_title("ROC Curves")
ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(fontsize=8)

# Precision-Recall
ax_pr = fig.add_subplot(gs[0,1])
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax_pr.plot(rec, prec, color=ACCENT, linewidth=2, label=f"AP={ap:.3f}")
ax_pr.axhline(y_test.mean(), color="#aaa", linestyle="--", label="Baseline")
ax_pr.set_title(f"Precision-Recall — {best_name}")
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision"); ax_pr.legend(fontsize=9)

# Confusion matrix
ax_cm = fig.add_subplot(gs[0,2])
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Injury","ACL Injury"]).plot(
    ax=ax_cm, colorbar=False, cmap="Blues")
ax_cm.set_title("Confusion Matrix")

# CV comparison bar
ax_cv = fig.add_subplot(gs[1,0])
names_ = list(results.keys()); aucs_ = [results[n]["cv_auc"] for n in names_]
stds_  = [results[n]["cv_std"] for n in names_]
colors_= [ACCENT if n==best_name else "#5dade2" for n in names_]
ax_cv.bar(names_, aucs_, yerr=stds_, capsize=5, color=colors_, edgecolor="white", width=0.5)
ax_cv.set_ylim(0,1); ax_cv.set_ylabel("AUC-ROC")
ax_cv.set_title("5-Fold CV AUC"); ax_cv.set_xticklabels(names_, rotation=15, ha="right")

# Feature importance
ax_fi = fig.add_subplot(gs[1,1:])
if hasattr(best_pipe["clf"], "feature_importances_"):
    importances = best_pipe["clf"].feature_importances_
else:
    r = permutation_importance(best_pipe, X_test, y_test, n_repeats=10, random_state=42)
    importances = r.importances_mean
fi_df = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True).tail(15)
fi_df.plot(kind="barh", ax=ax_fi, color=ACCENT, edgecolor="white")
ax_fi.set_title(f"Feature Importance — {best_name} (Top 15)")

fig.suptitle(f"OrthoAI ACL Prediction — Model Evaluation\nBest: {best_name} | Test AUC = {results[best_name]['test_auc']:.3f}",
             fontsize=14, fontweight="bold")
plt.show()


## 7. Discussion & Conclusions

### Key Findings

1. **ACWR is the strongest modifiable predictor.** Athletes with ACWR > 1.5
   (rapid training load spike) showed markedly higher injury rates — consistent with
   Gabbett (2016) and subsequent meta-analyses.

2. **Biomechanical factors matter independently.** Knee valgus angle and H:Q ratio
   contributed significantly after controlling for training load, supporting multi-modal
   screening protocols.

3. **Prior ACL injury is the strongest non-modifiable risk factor** — aligning with
   epidemiological data showing 4–6× increased re-injury risk.

4. **Model performance (AUC ~0.71) is within the range reported in published
   sports injury ML literature** (Claudino et al., 2019; Bartlett et al., 2022),
   suggesting the feature set captures meaningful signal despite the complexity of
   injury causation.

### Limitations

- Synthetic dataset: while calibrated to published parameter distributions,
  prospective validation on real cohort data is required before clinical deployment.
- Injury mechanism complexity: ACL injuries involve contact, non-contact, and
  terrain factors not fully captured by biomechanical screening alone.
- Class imbalance (~10% injured) limits precision; future work should explore
  SMOTE oversampling and probability calibration.

### Future Directions

- Apply pipeline to real open-access datasets (e.g., FIFA Medical Assessment Programme)
- Incorporate longitudinal GPS tracking data (chronic load calculated weekly)
- Develop a Streamlit clinical dashboard for practitioner-facing risk stratification

---

*This analysis was conducted as part of OrthoAI's independent AI research programme.*  
*Platform: OrthoAI Research | Founder: Muhammad Ebadur Rahman Siddiqui*
